<a href="https://colab.research.google.com/github/C-1Blue/ds2002-fa26/blob/main/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    for i in range(tries):
      r = requests.get(BASE, params=params, timeout=10)
          # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
      if r.status_code == 200:
        return pd.DataFrame(r.json()['daily'])
         # TODO:   raise on a status that is not in RETRY_STATUSES
      if r.status_code not in RETRY_STATUSES:
        r.raise_for_status()
            # TODO:   otherwise sleep (increasing each time) and try again
      if i < tries - 1:
        time.sleep(2 ** i)



    # TODO: raise if every attempt failed
    r.raise_for_status()
    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
df = get_weather(start='2026-09-05', end='2026-10-03')
#       then keep just the two dates you care about
target_dates = ['2026-09-05', '2026-10-03']
df = df[df['time'].isin(target_dates)].reset_index(drop=True)
used_fallback = False

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
if used_fallback:
  wx = FALLBACK
  wx['source'] = 'fallback sample'
else:
  wx = df[df['time'].isin(target_dates)].copy()
  wx['source'] = 'open-meteo'

wx = wx.reset_index(drop=True)

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO

wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
# Convert the date column to a real datetime type for clean joining
wx['date'] = pd.to_datetime(wx['date'])

# Reorder columns nicely (optional, but clean for output)
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
# Find the wetter and drier dates based on precipitation
wetter_row = wx.loc[wx['precip_mm'].idxmax()]
drier_row = wx.loc[wx['precip_mm'].idxmin()]
diff = wetter_row['precip_mm'] - drier_row['precip_mm']
date_wetter = pd.to_datetime(wetter_row['date']).strftime('%B %d, %Y')
date_drier = pd.to_datetime(drier_row['date']).strftime('%B %d, %Y')
print(f"{date_wetter} was wetter than {date_drier} by {diff:.1f} mm of precipitation.")

October 03, 2026 was wetter than September 05, 2026 by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
import datetime

# Define variables locally or retrieve defaults
lat = 38.03
lon = -78.51
start = '2026-09-05'
end = '2026-10-03'

pull_timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
data_origin = wx['source'].iloc[0] if 'source' in wx.columns else ('fallback sample' if used_fallback else 'open-meteo')

print(f"Endpoint: {BASE}")
print(f"Coordinates: {lat}, {lon}")
print(f"Date range: {start} to {end}")
print(f"Pulled at: {pull_timestamp}")
print(f"Data source: {data_origin}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-09 22:08:55
Data source: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

_If get_Weather silently returned as empty DataFrame, then a left join with sales data would fill all weather columsn with NaN values which causes the chart to render as blank due to plotting libraries dropping missing data. People won't catch it because the script run without raising errors._

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch

import pandas as pd


locations = {
    'Charlottesville': {'lat': 38.03, 'lon': -78.51},
    'Orlando': {'lat': 28.54, 'lon': -81.38}
}

target_dates = ['2026-09-05', '2026-10-03']
city_frames = []

for city_name, coords in locations.items():
    try:

        df = get_weather(start='2026-09-05', end='2026-10-03', lat=coords['lat'], lon=coords['lon'])
        wx_city = df[df['time'].isin(target_dates)].copy()
        wx_city['source'] = 'open-meteo'
    except Exception:

        wx_city = pd.DataFrame({
            'time': target_dates,
            'temperature_2m_max': [30.1, 18.4] if city_name == 'Charlottesville' else [32.5, 30.0],
            'precipitation_sum': [0.0, 22.5] if city_name == 'Charlottesville' else [2.1, 14.5],
            'source': ['fallback sample', 'fallback sample']
        })


    wx_city = wx_city.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm'
    })
    wx_city['date'] = pd.to_datetime(wx_city['date'])
    wx_city['city'] = city_name

    city_frames.append(wx_city)


long_wx = pd.concat(city_frames, ignore_index=True)
long_wx = long_wx[['city', 'date', 'temp_max', 'precip_mm', 'source']]